In [ ]:
# check available CUDA devices
import torch
devices = []
if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            device_name = torch.cuda.get_device_name(i)
            devices.append({
                "type": "CUDA",
                "available": True,
                "name": device_name,
                "index": i
            })
else:
    devices.append({"type": "CUDA", "available": False, "name": "N/A"})
devices

In [1]:
# change folder into the root of the ASR project
import os

if not os.path.isdir("Configs"):
    %cd ../

!pwd

/home/zathrus/stts2/AuxiliaryASR
/home/zathrus/stts2/AuxiliaryASR


In [2]:
# import packages, define common functions
import torch
import yaml
import os
from models import ASRCNN
from meldataset import build_dataloader
import torch.nn.functional as F
import pandas as pd
import os
import os.path as osp
from text_utils import TextCleaner
import itertools
from jiwer import wer
import re
from token_map import build_token_map_from_data

def cfg_get_nested(cfg: dict, path, default=None, sep="."):
    """
    Get a nested value from a dict using a list of keys or a dot-separated string.

    Examples:
        cfg_get_nested(config, ["model_params", "input_dim"], 80)
        cfg_get_nested(config, "model_params.input_dim", 80)
        cfg_get_nested(config, "top_key", 80)
    """
    if isinstance(path, str):
        keys = path.split(sep)
    else:
        keys = path

    cur = cfg
    for k in keys:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            return default
    return cur

def length_to_mask(lengths):
        mask = torch.arange(lengths.max()).unsqueeze(0).expand(lengths.shape[0], -1).type_as(lengths)
        mask = torch.gt(mask+1, lengths.unsqueeze(1))
        return mask

def load_ASR_models(ASR_MODEL_PATH, ASR_MODEL_CONFIG):
    def _load_model(model_config, model_path):
        model = ASRCNN(**model_config)
        params = torch.load(model_path, map_location='cpu', weights_only=False)['model']
        try:
            model.load_state_dict(params)
        except Exception as e:
            new_state_dict = {k.replace("module.", ""): v for k, v in params.items()}
            model.load_state_dict(new_state_dict)
        return model

    with open(ASR_MODEL_CONFIG) as f:
        config = yaml.safe_load(f)

    token_src = config.get('phoneme_maps_path')
    if not token_src:
        token_map = build_token_map_from_data(config.get('train_data'), config.get('val_data'), config.get('ood_data'), apply_asr_tokenizer=True)
    elif isinstance(token_src, dict):
        token_map = token_src
    else:
        csv = pd.read_csv(token_src, header=None).values
        token_map = {word: index for word, index in csv}
    
    model_params = cfg_get_nested( config, 'model_params', {
        'input_dim': 80,
        'hidden_dim': 256,
        'n_token': len( token_map ),
        'token_embedding_dim': 512,
        'n_layers': 5,
        'location_kernel_size': 31
    })

    if not 'n_token' in model_params:
        model_params['n_token'] = len( token_map )

    asr_model = _load_model(model_params, ASR_MODEL_PATH)

    return asr_model


In [3]:
checkpoint_dir = "Checkpoint_small_dataset_pr1_fixed_mel_bucketed"

files = [f for f in os.listdir( checkpoint_dir + "/") if f.startswith('epoch_') and f.endswith('.pth')]
sorted_files = sorted(files, key=lambda x: int(x.split('_')[-1].split('.')[0]))

model_path = checkpoint_dir + "/" + sorted_files[-1]
#model_path = "Checkpoint/epoch_00080.pth"

config_path = 'Checkpoint_small_dataset_pr1_fixed_mel_bucketed/config.yml'

config = yaml.safe_load(open(config_path))
phoneme_map = config.get('phoneme_maps_path')
if not phoneme_map:
    phoneme_map = build_token_map_from_data(config.get('train_data'), config.get('val_data'), config.get('ood_data'), apply_asr_tokenizer=True)

test_csv_path = config['val_data']

def _dict_desc(obj):
    return obj if isinstance(obj, str) else 'built from dataset'

print( "model --> " + model_path )
print( "config --> " + config_path)
print( "dictionary --> " + _dict_desc(phoneme_map))
print( "test: --> " + test_csv_path)

model = load_ASR_models(model_path, config_path)
model.eval()

print( "All OK ✓")


model --> Checkpoint_small_dataset_pr1_fixed_mel_bucketed/epoch_00160.pth
config --> Checkpoint_small_dataset_pr1_fixed_mel_bucketed/config.yml
dictionary --> Data/word_index_dict.txt
test: --> /home/zathrus/stts2/cs/results/final_filtered/val_list.txt
All OK ✓


In [4]:
text_cleaner = TextCleaner(phoneme_map)
device = "cpu"

with open(test_csv_path, "r", encoding="utf-8") as f:
    test_lines = f.readlines()

dataset_params = {
        'dict_path': cfg_get_nested( config, 'phoneme_maps_path', 'Data/word_index_dict.txt'),
        'sr': cfg_get_nested( config, 'preprocess_params.sr', 24000),
        'spect_params': cfg_get_nested( config, 'preprocess_params.spect_params', {
            'n_fft': 1024,
            'win_length': 1024,
            'hop_length': 300
        }),
        'mel_params': cfg_get_nested( config, 'preprocess_params.mel_params', { 'n_mels': 80 })
    }

test_loader = build_dataloader(
    path_list=[l[:-1].split('|') for l in test_lines],
    validation=True,
    batch_size=1,
    num_workers=2,
    device=device,
    collate_config={"return_wave": False},
    dataset_config=dataset_params
)

if isinstance(phoneme_map, str):
    csv = pd.read_csv(phoneme_map, header=None).values
    wlist = {word: index for word, index in csv}
else:
    wlist = phoneme_map
index2phoneme = {v: k for k, v in wlist.items()}

predictions = []
references = []
cleartexts = []

model.eval()
log_interval = 10
total = len(test_lines)
cntr = 0
maxtestsize = 1
#maxtestsize = 0

with torch.no_grad():
    for batch in test_loader:
        cleartexts.append(test_lines[cntr])

        texts, input_lengths, mels, output_lengths = batch  # from Collater

        mels = mels.to(device)
        output = model(mels)
        predicted_ids = torch.argmax(output, dim=-1)

        print(f"Batch {cntr} - Expected text: {test_lines[cntr].strip().split('|')[1]}")
        predicted_text = ''.join([text_cleaner.inverse_mapping.get(phoneme.item(), '<unk>') for phoneme in predicted_ids[0]])
        print(f"Batch {cntr} - Predicted text: {predicted_text}")

        for i in range(predicted_ids.size(0)):
            pred_seq = predicted_ids[i][:output_lengths[i]//3]
            ref_seq = texts[i][:input_lengths[i]]

            pred_phonemes = [index2phoneme.get(p.item(), '') for p in pred_seq]
            ref_phonemes = [index2phoneme.get(r.item(), '') for r in ref_seq]

            predictions.append(pred_phonemes)
            references.append(ref_phonemes)

        cntr += 1
        if (cntr)%log_interval == 0:
            print(f"{cntr} of {total} sentences tested")

        if maxtestsize > 0 and cntr >= maxtestsize:
            print(f"early stop reached at {maxtestsize} sentences")
            break

print("Done - all sententes tested.")


Options for MEL spectrogram calculations: {'sample_rate': 24000, 'n_mels': 80, 'n_fft': 1024, 'win_length': 1024, 'hop_length': 300}
Batch 0 - Expected text: mˈats mˈurphi zˈesebe nˈeviprˌaviː ˈaɲi slˈovo. «
Batch 0 - Predicted text:                                                        mˈˈamtssˌ     mˈurphi   zˈe  sse   be   nˈe vvii   prraa vvii   a niː    sslˈo  v               .              «               
early stop reached at 1 sentences
Done - all sententes tested.


In [ ]:
# Clean extra quotes and join tokens into space-separated strings
references_cleaned = [' '.join(token.strip('"') for token in seq) for seq in references]
predictions_cleaned = [' '.join(token.strip('"') for token in seq) for seq in predictions]

# Now use jiwer
per = wer(references_cleaned, predictions_cleaned)
print(f'Phoneme Error Rate: {per:.4f}')

In [5]:
###########################################
# Find the best AUX model (with best PER) #
###########################################

#checkpoint_dir = "Checkpoint-en-test"

files = [f for f in os.listdir( checkpoint_dir + "/") if f.startswith('epoch_') and f.endswith('.pth')]
sorted_files = sorted(files, key=lambda x: int(x.split('_')[-1].split('.')[0]))

model_path = checkpoint_dir + "/" + sorted_files[-1]
#model_path = "Checkpoint/epoch_00040.pth"
#config_path = "Configs/config.yml"

config = yaml.safe_load(open(config_path))
phoneme_map = config.get('phoneme_maps_path')
if not phoneme_map:
    phoneme_map = build_token_map_from_data(config.get('train_data'), config.get('val_data'), config.get('ood_data'), apply_asr_tokenizer=True)

test_csv_path = config['val_data']

def _dict_desc(obj):
    return obj if isinstance(obj, str) else 'built from dataset'

print( "config --> " + config_path )
print( "dictionary --> " + _dict_desc(phoneme_map) )
print( "test --> " + test_csv_path )

text_cleaner = TextCleaner(phoneme_map)
#text_cleaner = TextCleaner()
device = "cpu"

with open(test_csv_path, "r", encoding="utf-8") as f:
    test_lines = f.readlines()

dataset_params = {
        'dict_path': cfg_get_nested( config, 'phoneme_maps_path', 'Data/word_index_dict.txt'),
        'sr': cfg_get_nested( config, 'preprocess_params.sr', 24000),
        'spect_params': cfg_get_nested( config, 'preprocess_params.spect_params', {
            'n_fft': 1024,
            'win_length': 1024,
            'hop_length': 300
        }),
        'mel_params': cfg_get_nested( config, 'preprocess_params.mel_params', { 'n_mels': 80 })
    }

test_loader = build_dataloader(
    #path_list=test_lines,
    path_list=[l[:-1].split('|') for l in test_lines],
    validation=True,
    batch_size=1,
    num_workers=2,
    device=device,
    collate_config={"return_wave": False},
    dataset_config=dataset_params
)

if isinstance(phoneme_map, str):
    csv = pd.read_csv(phoneme_map, header=None).values
    wlist = {word: index for word, index in csv}
else:
    wlist = phoneme_map
index2phoneme = {v: k for k, v in wlist.items()}

best_model = ""
best_model_per = 100.0

model_cntr = 1
total_files = len(sorted_files)
results = []
#maxtestsize = 0 # will test the full validation set - may take a while and is usually not necessary
maxtestsize = 25

for aux_model_file in sorted_files:
    model_path = checkpoint_dir + "/" + aux_model_file
    model = load_ASR_models(model_path, config_path)
    model.eval()

    print(f"[{model_cntr}/{total_files}] Now evaluating AUX model: {aux_model_file}")
    model_cntr += 1

    predictions = []
    references = []
    first_ref = ""
    first_pred = ""

    test_iter = iter(test_loader)
    with torch.no_grad():
        for sample_idx in range(len(test_loader)):
            if maxtestsize > 0 and sample_idx >= maxtestsize:
                break

            batch = next(test_iter)
            texts, input_lengths, mels, output_lengths = batch
            mels = mels.to(device)
            output = model(mels)
            predicted_ids = torch.argmax(output, dim=-1)

            for i in range(predicted_ids.size(0)):
                pred_seq = predicted_ids[i][:output_lengths[i] // 3]
                ref_seq = texts[i][:input_lengths[i]]

                pred_phonemes = [index2phoneme.get(p.item(), '') for p in pred_seq]
                ref_phonemes = [index2phoneme.get(r.item(), '') for r in ref_seq]

                predictions.append(pred_phonemes)
                references.append(ref_phonemes)

                if first_ref == "":
                    first_ref = ' '.join(ref_phonemes)
                    first_pred = ' '.join(pred_phonemes)

    references_cleaned = [' '.join(seq) for seq in references]
    predictions_cleaned = [' '.join(seq) for seq in predictions]
    per = wer(references_cleaned, predictions_cleaned)

    print(f'Phoneme Error Rate: {per:.4f} {"✓" if per < best_model_per else "✗"}')
    if per < best_model_per:
        best_model_per = per
        best_model = aux_model_file

    results.append({
        'model': aux_model_file,
        'per': per,
        'first_ref': first_ref,
        'first_pred': first_pred
    })

results_sorted = sorted(results, key=lambda x: x['per'])

print("===================")
print("PERFORMANCE SUMMARY")
print("===================")
for res in results_sorted:
    print(f"Model: {res['model']}")
    print(f"PER: {res['per']:.4f}")
    print(f"Reference: {res['first_ref']}")
    print(f"Prediction: {res['first_pred']}")
    print("------------------------")

best = results_sorted[0]
print(f"✅ Best model: {best['model']} with PER = {best['per']:.4f}")


config --> Checkpoint_small_dataset_pr1_fixed_mel_bucketed/config.yml
dictionary --> Data/word_index_dict.txt
test --> /home/zathrus/stts2/cs/results/final_filtered/val_list.txt
Options for MEL spectrogram calculations: {'sample_rate': 24000, 'n_mels': 80, 'n_fft': 1024, 'win_length': 1024, 'hop_length': 300}
[1/20] Now evaluating AUX model: epoch_00010.pth
Phoneme Error Rate: 1.0000 ✓
[2/20] Now evaluating AUX model: epoch_00020.pth
Phoneme Error Rate: 0.7917 ✓
[3/20] Now evaluating AUX model: epoch_00030.pth
Phoneme Error Rate: 0.6181 ✓
[4/20] Now evaluating AUX model: epoch_00040.pth
Phoneme Error Rate: 0.6020 ✓
[5/20] Now evaluating AUX model: epoch_00050.pth
Phoneme Error Rate: 0.5715 ✓
[6/20] Now evaluating AUX model: epoch_00060.pth
Phoneme Error Rate: 0.5741 ✗
[7/20] Now evaluating AUX model: epoch_00070.pth
Phoneme Error Rate: 0.5639 ✓
[8/20] Now evaluating AUX model: epoch_00080.pth
Phoneme Error Rate: 0.5986 ✗
[9/20] Now evaluating AUX model: epoch_00090.pth
Phoneme Error Ra